# Mastering LLM Deployment
## Day 1 · Lab 2 — Model Distillation

**Duration:** ~90 minutes  ·  **Runtime required:** T4 GPU  ·  **Prerequisite:** Lab 1 completed (the ledger and lab kit exist)

---

### Why distillation comes first

In the Lab 1 case study, the reported playbook applied three model-level levers. Distillation is the one to attempt first, for a structural reason:

**Quantization and pruning modify a fixed architecture. Distillation changes the architecture itself.** Cutting a 12-layer encoder to 4 layers removes two-thirds of the transformer compute permanently, and the result is *still a normal fp32 model* — so it can then be quantized and pruned on top. The levers compose in that direction; they do not compose as well in reverse.

### What you will build

Following the syllabus, we distil on **SQuAD v1.1** extractive question answering rather than on a classification task. That is a deliberately harder setting, and it is the more useful one to learn:

- The output is not one label but **two probability distributions over token positions** (answer start and answer end). The distillation loss has to handle structured output.
- Answer-span prediction degrades in visible, diagnosable ways when a student is too small — you can read the failure, not just the metric.
- The preprocessing (context windowing, offset mapping, span alignment) is exactly the preprocessing you will re-implement in any production QA or extraction service.

### Learning outcomes

- Explain the mechanism of knowledge distillation: soft targets, temperature, and what "dark knowledge" actually refers to.
- Construct a compact student by truncating and initialising from a trained teacher.
- Implement a distillation loss over structured (start/end span) outputs, with padding masked correctly.
- Precompute teacher logits for **offline distillation** and explain when that trade is worth making.
- Measure the size / latency / quality triangle and add a second row to the optimization ledger.
- Package reusable KD utilities that Lab 5 will apply to the Lab 1 sentiment model.

**Expected GPU time: 8–12 minutes.**

---
## 0. Environment setup

Identical to Lab 1. If you are continuing in the same session and have already run Lab 1, you still need to run these cells — a fresh notebook is a fresh Python process.

In [ ]:
%pip install -q "transformers>=4.40,<5" "datasets>=2.19,<4" "tf-keras>=2.16" "tensorflow-model-optimization>=0.8.0" "scikit-learn" "pandas"
print('dependencies installed')

In [ ]:
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

if "tensorflow" in sys.modules:
    print("TensorFlow already imported -> Runtime > Restart session and re-run from the top.")

import tensorflow as tf, numpy as np, transformers, time
# tf.keras is a lazy loader and does not re-export __version__.
try:
    import tf_keras as _keras_pkg
except ImportError:
    import keras as _keras_pkg
_keras_impl = tf.keras.Model.__module__
print("TF", tf.__version__, "| Keras", _keras_pkg.__version__, "|", _keras_impl,
      "| transformers", transformers.__version__)
assert _keras_pkg.__version__.startswith("2.") and "tf_keras" in _keras_impl, (
    "Keras 2 is not active. Install tf-keras, set TF_USE_LEGACY_KERAS=1 before "
    "importing tensorflow, then Runtime > Restart session.")
tf.keras.utils.set_random_seed(42)

In [ ]:
USE_DRIVE = True
ROOT = "/content/llm-deploy-labs"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/llm-deploy-labs"
    except Exception as e:
        print("Drive unavailable:", e)
os.environ["LLMDEPLOY_ROOT"] = ROOT
for sub in ("models", "reports", "data"):
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
print("Artifact root:", ROOT)

In [ ]:

LABKIT_SRC = r'''
"""
labkit.py - shared utilities for the "Mastering LLM Deployment" hands-on labs.

Everything the labs need in common lives here so that each notebook measures
the same things in the same way:

  * artifact + ledger management (results survive across notebooks via Drive)
  * a model "size on disk" and parameter/sparsity accounting
  * a latency/throughput benchmark harness with warm-up and percentiles
  * a minimal, explicit GradientTape training loop (works for HF TF models,
    plain Keras models, distillation losses and masked/pruned training alike)
"""

import os

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import json
import shutil
import time
from pathlib import Path

import numpy as np
import tensorflow as tf

# --------------------------------------------------------------------------
# 1. Artifact root
# --------------------------------------------------------------------------

_ROOT = Path(os.environ.get("LLMDEPLOY_ROOT", "/content/llm-deploy-labs"))


def set_root(path):
    """Point the lab kit at a persistent directory (ideally on Google Drive)."""
    global _ROOT
    _ROOT = Path(path)
    for sub in ("models", "reports", "data"):
        (_ROOT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["LLMDEPLOY_ROOT"] = str(_ROOT)
    return _ROOT


def root():
    return _ROOT


def model_dir(name, clean=False):
    """Return (and create) a directory under <root>/models/<name>."""
    d = _ROOT / "models" / name
    if clean and d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    return d


# --------------------------------------------------------------------------
# 2. Size and parameter accounting
# --------------------------------------------------------------------------


def size_mb(path):
    """Size of a file or, recursively, of a directory - in MB."""
    p = Path(path)
    if p.is_file():
        return p.stat().st_size / 1e6
    total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    return total / 1e6


def count_params(model):
    """Total trainable parameter count."""
    return int(sum(int(np.prod(v.shape)) for v in model.trainable_variables))


def weight_sparsity(model, kinds=("kernel", "weight", "embeddings")):
    """Fraction of zeros across the "real" weight matrices (ignores biases /
    LayerNorm, which are never pruned in practice)."""
    zeros, total = 0, 0
    for v in model.trainable_variables:
        if not any(k in v.name for k in kinds):
            continue
        arr = v.numpy()
        zeros += int((arr == 0).sum())
        total += int(arr.size)
    return zeros / max(total, 1)


# --------------------------------------------------------------------------
# 3. Latency / throughput benchmarking
# --------------------------------------------------------------------------


def measure_latency(predict_fn, inputs, warmup=5, runs=30, batch_size=1):
    """Run predict_fn(inputs) repeatedly and report wall-clock percentiles.

    Warm-up matters: the first calls pay for graph tracing, kernel autotuning
    and (on GPU) cuDNN algorithm selection. Reporting those numbers is the
    single most common benchmarking mistake in deployment work.
    """
    for _ in range(warmup):
        predict_fn(inputs)

    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        predict_fn(inputs)
        samples.append((time.perf_counter() - t0) * 1000.0)

    samples = np.array(sorted(samples))
    p50 = float(np.percentile(samples, 50))
    return {
        "mean_ms": round(float(samples.mean()), 2),
        "p50_ms": round(p50, 2),
        "p90_ms": round(float(np.percentile(samples, 90)), 2),
        "p95_ms": round(float(np.percentile(samples, 95)), 2),
        "throughput_rps": round(batch_size / (p50 / 1000.0), 1),
    }


def device_label():
    return "GPU" if tf.config.list_physical_devices("GPU") else "CPU"


# --------------------------------------------------------------------------
# 4. The optimization ledger
# --------------------------------------------------------------------------


def _ledger_file():
    (_ROOT / "reports").mkdir(parents=True, exist_ok=True)
    return _ROOT / "reports" / "ledger.json"


def load_ledger():
    f = _ledger_file()
    if not f.exists():
        return []
    return json.loads(f.read_text())


def record(stage, **fields):
    """Insert or replace a ledger row. Stage names are unique keys, so
    re-running a cell updates the row instead of duplicating it."""
    ledger = [e for e in load_ledger() if e.get("stage") != stage]
    entry = {"stage": stage, "recorded_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    entry.update(fields)
    ledger.append(entry)
    _ledger_file().write_text(json.dumps(ledger, indent=2))
    return entry


def ledger_df(columns=None):
    import pandas as pd

    df = pd.DataFrame(load_ledger())
    if df.empty:
        return df
    preferred = [
        "stage",
        "model",
        "task",
        "dataset",
        "params_m",
        "size_mb",
        "quality",
        "quality_metric",
        "p50_ms",
        "p95_ms",
        "throughput_rps",
        "device",
        "notes",
    ]
    cols = columns or [c for c in preferred if c in df.columns]
    extra = [c for c in df.columns if c not in cols and c != "recorded_at"]
    return df[cols + extra]


# --------------------------------------------------------------------------
# 5. A small, explicit training loop
# --------------------------------------------------------------------------


def train(
    model,
    dataset,
    loss_fn,
    optimizer,
    epochs=1,
    steps_per_epoch=None,
    log_every=50,
    on_step_end=None,
    clip_norm=1.0,
):
    """Generic GradientTape loop.

    loss_fn(model, batch, training) -> scalar loss tensor.
    on_step_end(global_step) -> optional Python callback, used by the pruning
    lab to update sparsity masks between steps.
    """

    @tf.function
    def train_step(batch):
        with tf.GradientTape() as tape:
            loss = loss_fn(model, batch, True)
        grads = tape.gradient(loss, model.trainable_variables)
        pairs = [
            (g, v) for g, v in zip(grads, model.trainable_variables) if g is not None
        ]
        if clip_norm:
            gs, _ = tf.clip_by_global_norm([g for g, _ in pairs], clip_norm)
            pairs = list(zip(gs, [v for _, v in pairs]))
        optimizer.apply_gradients(pairs)
        return loss

    global_step = 0
    history = []
    for epoch in range(epochs):
        running, seen = 0.0, 0
        t0 = time.time()
        for step, batch in enumerate(dataset):
            loss = float(train_step(batch))
            running += loss
            seen += 1
            global_step += 1
            if on_step_end is not None:
                on_step_end(global_step)
            if log_every and global_step % log_every == 0:
                print(
                    f"  epoch {epoch + 1} | step {global_step:>5} | "
                    f"loss {running / seen:.4f}"
                )
                running, seen = 0.0, 0
            if steps_per_epoch and step + 1 >= steps_per_epoch:
                break
        history.append({"epoch": epoch + 1, "seconds": round(time.time() - t0, 1)})
        print(f"  epoch {epoch + 1} finished in {history[-1]['seconds']}s")
    return history


# --------------------------------------------------------------------------
# 6. Evaluation helpers
# --------------------------------------------------------------------------


def evaluate_accuracy(logits_fn, dataset):
    """logits_fn(features) -> array of shape [batch, num_classes]."""
    correct, total = 0, 0
    for features, labels in dataset:
        logits = np.asarray(logits_fn(features))
        preds = logits.argmax(axis=-1)
        labels = np.asarray(labels)
        correct += int((preds == labels).sum())
        total += int(labels.shape[0])
    return correct / max(total, 1)


def hf_logits_fn(model):
    """Wrap a Hugging Face TF model so it returns a plain logits tensor and is
    compiled once into a graph (fair, low-overhead benchmarking)."""

    @tf.function(reduce_retracing=True)
    def fn(features):
        return model(features, training=False).logits

    return fn


def banner(title):
    line = "=" * max(60, len(title) + 4)
    print(f"\n{line}\n  {title}\n{line}")
'''

from pathlib import Path
Path(ROOT, 'labkit.py').write_text(LABKIT_SRC)

import sys, importlib
sys.path.insert(0, ROOT)
import labkit as lk
importlib.reload(lk)
lk.set_root(ROOT)
lk.banner('lab kit ready')
print('root  :', lk.root())
print('device:', lk.device_label())

---
## 1. How distillation works

### 1.1 The core idea

A trained classifier produces a full probability distribution, not just an argmax. For a question-answering model asked where an answer starts, the distribution over token positions might put 0.7 on the correct token, 0.2 on the token immediately before it, 0.05 on the start of a plausible but wrong span, and near-zero elsewhere.

The hard label says only "position 41". The distribution additionally says *position 40 is nearly as good, position 12 is a reasonable distractor, and position 3 is absurd*. That ranking over wrong answers is the teacher's learned similarity structure — commonly called **dark knowledge** — and it is a far richer training signal per example than a one-hot target.

Distillation trains the student to match that distribution.

### 1.2 Temperature

Softmax with temperature $T$:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T = 1$ is the ordinary softmax. A confident teacher produces a near-one-hot distribution, which carries almost no more information than the hard label.
- $T > 1$ flattens the distribution, amplifying the relative structure among the low-probability classes — which is precisely the signal we want to transfer.
- $T \to \infty$ approaches uniform, and the signal disappears into noise.

Typical values are 2 to 5. Because gradients of the soft-target term scale as $1/T^2$, the term is multiplied by $T^2$ to keep its magnitude comparable to the hard-label term when $T$ changes.

### 1.3 The combined loss

$$\mathcal{L} = \alpha \cdot T^2 \cdot \mathrm{CE}\big(\sigma(z_t/T),\ \sigma(z_s/T)\big) \;+\; (1-\alpha) \cdot \mathrm{CE}\big(y,\ \sigma(z_s)\big)$$

The first term transfers the teacher's behaviour. The second keeps the student anchored to ground truth, which matters because the teacher is wrong on some examples and you do not want the student to faithfully reproduce those errors. $\alpha$ between 0.5 and 0.9 is typical.

### 1.4 What DistilBERT added

The DistilBERT recipe is the reference point for the technique on transformers, and three of its choices are worth copying:

1. **Halve the depth, keep the width.** Hidden size stays at 768; layer count drops from 12 to 6. Depth reduction cuts latency roughly linearly; width reduction hurts quality faster for the same saving, and breaks weight-copy initialisation.
2. **Initialise the student from the teacher's layers.** Take every second layer rather than the first *n*. A student initialised this way starts near a good solution instead of from scratch.
3. **Add a cosine embedding loss** aligning student and teacher hidden states, in addition to the output distributions. We discuss this below but keep the lab's loss to output-level distillation, which is where most of the benefit is and which generalises to any teacher you can only query.

### 1.5 Task-agnostic vs task-specific

| | Distil first, then fine-tune | Fine-tune first, then distil |
|---|---|---|
| **What transfers** | general language ability | task behaviour |
| **Cost** | very high (pretraining-scale) | low (one task dataset) |
| **Reuse** | one student serves many tasks | one student per task |
| **Best when** | you own many downstream tasks | you are shipping one endpoint |

We do the second: our teacher is already fine-tuned on SQuAD, and we distil its task behaviour into a smaller student. This is the common industrial case and it is what fits in a lab.

---
## 2. Load the teacher

We use a publicly available BERT-base checkpoint already fine-tuned on SQuAD v1.1, which saves the 40+ minutes it would take to fine-tune one here. The weights are PyTorch, so `from_pt=True` converts them into the TensorFlow model on load.

This is also a realistic scenario: in production distillation work, the teacher is usually an existing artifact you did not train.

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForQuestionAnswering, BertConfig

TEACHER_ID = "csarron/bert-base-uncased-squad-v1"

tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)
def load_tf_pretrained(cls, name_or_path, **kwargs):
    # Robust TF-model loader.
    #
    # transformers' PyTorch -> TensorFlow converter iterates the state dict
    # directly. Newer safetensors hands it a `safe_open` handle instead of a
    # dict, which is not iterable, so the conversion dies with:
    #     TypeError: 'builtins.safe_open' object is not iterable
    #
    # Two-step strategy:
    #   1. Ask for non-safetensors weights. Most Hub repos still ship
    #      tf_model.h5 or pytorch_model.bin, and either avoids the bug.
    #   2. If only safetensors exist, materialise them into a real .bin on
    #      disk and convert from that, which takes the working code path.
    try:
        return cls.from_pretrained(name_or_path, use_safetensors=False, **kwargs)
    except Exception as first_error:
        print("[loader] direct load failed:", type(first_error).__name__, first_error)
        print("[loader] falling back to manual safetensors conversion")

    import os, glob, shutil, tempfile, torch
    from safetensors.torch import load_file
    from huggingface_hub import snapshot_download

    src = (name_or_path if os.path.isdir(name_or_path)
           else snapshot_download(
               name_or_path,
               allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors"]))

    shards = sorted(glob.glob(os.path.join(src, "*.safetensors")))
    if not shards:
        raise RuntimeError(f"no safetensors weights found in {src}")

    dst = tempfile.mkdtemp(prefix="tfconv-")
    for fname in os.listdir(src):
        if fname.endswith((".json", ".txt", ".model")) and "index" not in fname:
            shutil.copy(os.path.join(src, fname), dst)

    state = {}
    for shard in shards:
        state.update(load_file(shard))
    torch.save(state, os.path.join(dst, "pytorch_model.bin"))
    print(f"[loader] converted {len(shards)} shard(s), {len(state)} tensors -> pytorch_model.bin")

    return cls.from_pretrained(dst, from_pt=True, **kwargs)


teacher = load_tf_pretrained(TFAutoModelForQuestionAnswering, TEACHER_ID)

teacher.trainable = False
print("teacher layers    :", teacher.config.num_hidden_layers)
print("teacher hidden    :", teacher.config.hidden_size)
print("teacher parameters:", f"{teacher.num_parameters()/1e6:.1f}M")

### 2.1 Inspect the teacher's output distribution

Before writing any loss, look at what we are actually transferring. The cell below runs one question through the teacher and shows the top start-position candidates at three temperatures.

Watch what temperature does: at $T=1$ the distribution is nearly one-hot and tells the student little beyond the label; at $T=4$ the runners-up become visible. Those runners-up are the signal.

In [ ]:
context  = ("The Amazon rainforest is a moist broadleaf forest that covers most of the "
            "Amazon basin of South America. This basin encompasses 7,000,000 square "
            "kilometres, of which 5,500,000 square kilometres are covered by the rainforest. "
            "The majority of the forest is contained within Brazil, with 60 percent of the "
            "rainforest, followed by Peru with 13 percent.")
question = "Which country contains the majority of the Amazon rainforest?"

enc = tokenizer(question, context, return_tensors="tf", max_length=384,
                truncation="only_second", padding="max_length")
out = teacher(enc, training=False)
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0].numpy())

start_logits = out.start_logits[0].numpy()
for T in (1.0, 2.0, 4.0):
    probs = tf.nn.softmax(start_logits / T).numpy()
    top = np.argsort(probs)[::-1][:5]
    formatted = ", ".join(f"{tokens[i]}={probs[i]:.3f}" for i in top)
    print(f"T={T:<4} top-5 start positions: {formatted}")

best_s, best_e = int(start_logits.argmax()), int(out.end_logits[0].numpy().argmax())
print("\npredicted answer:",
      tokenizer.decode(enc["input_ids"][0][best_s:best_e + 1]))

---
## 3. Prepare SQuAD

### 3.1 The preprocessing problem

Extractive QA preprocessing is where most implementation bugs live, so it is worth being explicit about what has to happen:

1. Encode `question` and `context` as a **pair**, truncating **only the context** (`truncation="only_second"`) — never the question.
2. A context longer than the window must be split into **overlapping chunks** (`stride`), because the answer might straddle a naive cut point. One example can therefore produce several features.
3. The labels are **token indices**, but the dataset gives **character offsets**. We use `return_offsets_mapping=True` to translate, and `sequence_ids()` to know which tokens belong to the context rather than the question or the special tokens.
4. If the answer is not inside the current chunk, the label points at the `[CLS]` token — the conventional "no answer here" target.

We build features with plain Python rather than `datasets.map` so every step is visible and debuggable.

In [ ]:
from datasets import load_dataset

def load_hf_dataset(repo_id, config=None, **kwargs):
    # Robust dataset loader.
    #
    # Hub dataset metadata is periodically regenerated with whatever `datasets`
    # version is current. A repo written by datasets 4.x declares the `List`
    # feature type, which 2.x/3.x cannot parse:
    #     ValueError: Feature type 'List' not found.
    #
    # The underlying parquet files are unaffected, so the fallback skips the
    # metadata entirely and reads the parquet shards directly.
    from datasets import load_dataset
    try:
        return (load_dataset(repo_id, config, **kwargs) if config
                else load_dataset(repo_id, **kwargs))
    except Exception as err:
        print("[data] standard load failed:", type(err).__name__, err)
        print("[data] falling back to direct parquet load")

    import os, re, collections
    from huggingface_hub import list_repo_files, hf_hub_download
    from datasets import load_dataset

    files = [f for f in list_repo_files(repo_id, repo_type="dataset")
             if f.endswith(".parquet")]
    if not files:
        raise RuntimeError(f"no parquet shards found in {repo_id}")

    if config:
        scoped = [f for f in files if f.split("/")[0] == config]
        if scoped:
            files = scoped
    else:
        dirs = {f.split("/")[0] for f in files if "/" in f}
        if len(dirs) > 1:
            pick = "plain_text" if "plain_text" in dirs else sorted(dirs)[0]
            print(f"[data] multiple configs {sorted(dirs)}; using {pick!r}")
            files = [f for f in files if f.split("/")[0] == pick]

    shards = collections.defaultdict(list)
    for f in files:
        stem = os.path.basename(f)
        m = re.match(r"([A-Za-z0-9_.\-]+?)(?:-\d+-of-\d+)?\.parquet$", stem)
        shards[m.group(1) if m else "train"].append(
            hf_hub_download(repo_id, f, repo_type="dataset"))

    print("[data] splits:", {k: len(v) for k, v in sorted(shards.items())})
    return load_dataset("parquet",
                        data_files={k: sorted(v) for k, v in shards.items()})


MAX_LEN     = 384
DOC_STRIDE  = 128
N_TRAIN     = 10_000      # raise for better students; 10k keeps the lab inside its budget
N_EVAL      = 1_000

squad = load_hf_dataset("rajpurkar/squad")
train_examples = squad["train"].shuffle(seed=42).select(range(N_TRAIN))
eval_examples  = squad["validation"].select(range(N_EVAL))
print(f"train examples {len(train_examples):,} | eval examples {len(eval_examples):,}")
print("\nan example:")
ex = train_examples[0]
print("  Q:", ex["question"])
print("  A:", ex["answers"]["text"][0], "at char", ex["answers"]["answer_start"][0])

In [ ]:
def tokenize_pairs(examples):
    return tokenizer(
        [q.lstrip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )


def build_train_features(examples):
    # Map character-level answers onto token start/end indices.
    tok = tokenize_pairs(examples)
    sample_map = tok.pop("overflow_to_sample_mapping")
    offsets_all = tok.pop("offset_mapping")
    answers_col = examples["answers"]
    starts, ends = [], []
    for i, offsets in enumerate(offsets_all):
        input_ids = tok["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        seq_ids = tok.sequence_ids(i)
        answers = answers_col[sample_map[i]]

        if len(answers["answer_start"]) == 0:
            starts.append(cls_index); ends.append(cls_index); continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        ts = 0
        while seq_ids[ts] != 1:
            ts += 1
        te = len(input_ids) - 1
        while seq_ids[te] != 1:
            te -= 1

        # Answer not fully inside this window -> point at [CLS]
        if not (offsets[ts][0] <= start_char and offsets[te][1] >= end_char):
            starts.append(cls_index); ends.append(cls_index)
        else:
            while ts < len(offsets) and offsets[ts][0] <= start_char:
                ts += 1
            starts.append(ts - 1)
            while offsets[te][1] >= end_char:
                te -= 1
            ends.append(te + 1)

    feats = {k: np.asarray(tok[k], dtype=np.int32)
             for k in ("input_ids", "attention_mask", "token_type_ids")}
    return feats, np.asarray(starts, np.int32), np.asarray(ends, np.int32)


def build_eval_features(examples):
    # Keep offset mappings (context tokens only) so predictions can be
    # mapped back to character spans in the original context.
    tok = tokenize_pairs(examples)
    sample_map = tok.pop("overflow_to_sample_mapping")
    offsets_all = tok.pop("offset_mapping")
    ids_col = examples["id"]
    example_ids, kept_offsets = [], []
    for i, offsets in enumerate(offsets_all):
        seq_ids = tok.sequence_ids(i)
        example_ids.append(ids_col[sample_map[i]])
        kept_offsets.append([o if seq_ids[k] == 1 else None
                             for k, o in enumerate(offsets)])

    feats = {k: np.asarray(tok[k], dtype=np.int32)
             for k in ("input_ids", "attention_mask", "token_type_ids")}
    return feats, example_ids, kept_offsets


train_feats, train_starts, train_ends = build_train_features(train_examples)
eval_feats, eval_example_ids, eval_offsets = build_eval_features(eval_examples)

print(f"train: {len(train_examples):,} examples -> {len(train_starts):,} features "
      f"({len(train_starts)/len(train_examples):.2f} windows per example)")
print(f"eval : {len(eval_examples):,} examples -> {len(eval_example_ids):,} features")
print(f"{int((train_starts == 0).sum())} training features have the answer outside "
      f"their window (labelled [CLS])")

---
## 4. Precompute the teacher's logits — offline distillation

There are two ways to run distillation:

| | **Online** (teacher in the loop) | **Offline** (logits precomputed) |
|---|---|---|
| Teacher forward passes | once per epoch per example | once, total |
| GPU memory | both models resident | student only |
| Supports data augmentation | yes | no (augmented inputs have no cached logits) |
| Supports a teacher you can only call remotely | awkward | naturally |

Offline is strictly cheaper when the training data is fixed, and it scales to the case where the teacher is an API you pay per call. We use it here: one pass over 10k features, then the teacher is no longer needed on the GPU.

The cached logits are `[num_features, 384]` floats for each of start and end — about 30 MB. For a large corpus this becomes the constraint that pushes you back to online distillation, or to caching top-k logits only.

**Expected time: 60–90 seconds.**

In [ ]:
def teacher_logits_over(features, batch_size=32):
    n = features["input_ids"].shape[0]
    starts, ends = [], []
    for i in range(0, n, batch_size):
        batch = {k: tf.constant(v[i:i + batch_size]) for k, v in features.items()}
        out = teacher(batch, training=False)
        starts.append(out.start_logits.numpy())
        ends.append(out.end_logits.numpy())
        if (i // batch_size) % 50 == 0:
            print(f"  {min(i + batch_size, n):>6}/{n}")
    return np.concatenate(starts), np.concatenate(ends)

lk.banner("caching teacher logits over the training features")
t0 = time.time()
t_start_logits, t_end_logits = teacher_logits_over(train_feats)
print(f"done in {time.time() - t0:.0f}s | cached array shape {t_start_logits.shape} "
      f"| {(t_start_logits.nbytes + t_end_logits.nbytes)/1e6:.0f} MB")

In [ ]:
BATCH_SIZE = 16

train_ds = tf.data.Dataset.from_tensor_slices((
    train_feats,
    {"start": train_starts, "end": train_ends,
     "t_start": t_start_logits, "t_end": t_end_logits},
)).shuffle(2048, seed=42).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(train_ds.element_spec[1]["t_start"])

---
## 5. Build the student

### 5.1 Architecture choice

We keep the hidden size at 768 and cut depth from 12 layers to 4. Two reasons:

- **Depth reduction is the cleanest latency lever.** A transformer's per-token cost is roughly linear in layer count, so 12 → 4 predicts roughly a 3× speedup in the encoder.
- **Keeping the width lets us copy weights.** A student with a different hidden size cannot be initialised from teacher layers, and would have to learn everything — including the embedding table — from the distillation signal alone.

Note what this does *not* shrink: the embedding matrix is 30,522 × 768 ≈ 23.4M parameters, and it is unaffected by depth. That is why a 4-layer student is not one-third the size of a 12-layer teacher. Embedding-table size is a separate problem, addressed by vocabulary reduction or embedding factorisation.

### 5.2 Which layers to copy

We take teacher layers **2, 5, 8, 11** — evenly spaced, including the last. Copying the *first* four layers gives a student that has only ever seen shallow, largely syntactic representations; even spacing preserves the progression from surface features to task-specific ones.

In [ ]:
from transformers import TFBertForQuestionAnswering

LAYER_MAP = [2, 5, 8, 11]     # teacher layer index -> student layer 0..3

student_config = BertConfig.from_pretrained(TEACHER_ID)
student_config.num_hidden_layers = len(LAYER_MAP)
student = TFBertForQuestionAnswering(student_config)

# A subclassed TF model must be called once before its weights exist.
dummy = {k: tf.constant(v[:1]) for k, v in train_feats.items()}
_ = student(dummy, training=False)
_ = teacher(dummy, training=False)

copied = []
try:
    student.bert.embeddings.set_weights(teacher.bert.embeddings.get_weights())
    copied.append("embeddings")
    for s_idx, t_idx in enumerate(LAYER_MAP):
        student.bert.encoder.layer[s_idx].set_weights(
            teacher.bert.encoder.layer[t_idx].get_weights())
        copied.append(f"encoder.layer[{s_idx}] <- teacher[{t_idx}]")
    student.qa_outputs.set_weights(teacher.qa_outputs.get_weights())
    copied.append("qa_outputs")
    print("initialised from teacher:")
    for c in copied:
        print("  ", c)
except Exception as e:
    print("Weight copy failed, student starts from random init:", e)
    print("The lab still runs, but expect noticeably lower F1.")

t_params, s_params = teacher.num_parameters(), student.num_parameters()
print(f"\nteacher {t_params/1e6:6.1f}M parameters")
print(f"student {s_params/1e6:6.1f}M parameters  "
      f"({s_params/t_params:.0%} of the teacher)")
emb = student_config.vocab_size * student_config.hidden_size
print(f"of which the embedding table alone is {emb/1e6:.1f}M "
      f"({emb/s_params:.0%} of the student)")

---
## 6. The distillation loss for span prediction

Two design points that are easy to get wrong:

**Mask the padding.** Softmax runs over all 384 positions, most of which are `[PAD]`. Teacher logits at padded positions are arbitrary garbage; letting them into the distribution wastes probability mass and injects noise. We add a large negative value at masked positions before every softmax, for teacher and student alike.

**Treat start and end symmetrically.** Both are distributions over the same 384 positions, so we apply the identical loss to each and average. A model that gets starts right and ends wrong produces spans that are individually plausible and jointly useless.

We write the soft-target term as a cross-entropy between the teacher's soft distribution and the student's log-probabilities. It differs from KL divergence only by the teacher's entropy, which is a constant with respect to the student's parameters — so the gradients are identical, and this form is cheaper.

In [ ]:
TEMPERATURE = 3.0
ALPHA       = 0.7          # weight on the soft-target (teacher) term
NEG_INF     = -1e4

hard_loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction=tf.keras.losses.Reduction.NONE)


def soft_cross_entropy(teacher_logits, student_logits, mask, T):
    # CE( softmax(teacher/T), softmax(student/T) ) with padding masked out.
    penalty = (1.0 - tf.cast(mask, tf.float32)) * (-NEG_INF)   # positive amount to subtract
    t_masked = teacher_logits - penalty
    s_masked = student_logits - penalty
    t_probs = tf.nn.softmax(t_masked / T, axis=-1)
    s_logp  = tf.nn.log_softmax(s_masked / T, axis=-1)
    return -tf.reduce_sum(t_probs * s_logp, axis=-1)


def distillation_loss(model, batch, training):
    features, targets = batch
    mask = features["attention_mask"]
    out = model(features, training=training)

    soft = 0.5 * (
        soft_cross_entropy(targets["t_start"], out.start_logits, mask, TEMPERATURE)
        + soft_cross_entropy(targets["t_end"], out.end_logits, mask, TEMPERATURE)
    )
    hard = 0.5 * (
        hard_loss_fn(targets["start"], out.start_logits)
        + hard_loss_fn(targets["end"], out.end_logits)
    )
    total = ALPHA * (TEMPERATURE ** 2) * soft + (1.0 - ALPHA) * hard
    return tf.reduce_mean(total)


# Sanity check on one batch before committing to a training run.
probe = next(iter(train_ds))
print("initial loss:", float(distillation_loss(student, probe, False)))

### 6.1 Train the student

**Expected time on a T4: 10–20 minutes** for 2 epochs over ~10k features.

In [ ]:
EPOCHS = 2
LR     = 5e-5

steps = int(np.ceil(len(train_starts) / BATCH_SIZE)) * EPOCHS
sched = tf.keras.optimizers.schedules.PolynomialDecay(LR, steps, end_learning_rate=0.0)
opt   = tf.keras.optimizers.Adam(learning_rate=sched)

lk.banner(f"distilling {len(LAYER_MAP)}-layer student | {steps} steps")
lk.train(student, train_ds, distillation_loss, opt, epochs=EPOCHS, log_every=150)

---
## 7. Evaluate: exact match and F1

SQuAD is scored with two metrics, and we implement both rather than pulling a metric package — the logic is short and worth seeing.

- **Exact Match (EM):** the normalised prediction equals a normalised gold answer. Binary, harsh.
- **F1:** token-level overlap between prediction and gold. Partial credit — "Brazil" against "within Brazil" scores well on F1 and zero on EM.

Normalisation lowercases, strips articles and punctuation, and collapses whitespace. Each question has several gold answers; we score against the best one.

Turning start/end logits into a text span requires the inverse of the preprocessing: search the top-k start and end positions, reject spans that are inverted, too long, or outside the context, take the highest-scoring survivor, and use the offset mapping to slice the *original* context string. Never detokenize — `##` fragments and lost whitespace will silently cost you F1.

In [ ]:
import re, string, collections

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    common = collections.Counter(p) & collections.Counter(g)
    overlap = sum(common.values())
    if len(p) == 0 or len(g) == 0:
        return float(p == g)
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)


def postprocess(examples, example_ids, offsets, start_logits, end_logits,
                n_best=20, max_answer_len=30):
    by_example = collections.defaultdict(list)
    for feat_idx, ex_id in enumerate(example_ids):
        by_example[ex_id].append(feat_idx)

    predictions = {}
    for ex in examples:
        context, best_score, best_text = ex["context"], -1e9, ""
        for fi in by_example[ex["id"]]:
            s_log, e_log, offs = start_logits[fi], end_logits[fi], offsets[fi]
            s_cand = np.argsort(s_log)[-n_best:][::-1]
            e_cand = np.argsort(e_log)[-n_best:][::-1]
            for s in s_cand:
                for e in e_cand:
                    if s >= len(offs) or e >= len(offs):
                        continue
                    if offs[s] is None or offs[e] is None:
                        continue
                    if e < s or (e - s + 1) > max_answer_len:
                        continue
                    score = s_log[s] + e_log[e]
                    if score > best_score:
                        best_score = score
                        best_text = context[offs[s][0]:offs[e][1]]
        predictions[ex["id"]] = best_text
    return predictions


def score(predictions, examples):
    em_total = f1_total = 0.0
    for ex in examples:
        pred = predictions[ex["id"]]
        golds = ex["answers"]["text"]
        em_total += max(exact_match(pred, g) for g in golds)
        f1_total += max(f1(pred, g) for g in golds)
    n = len(examples)
    return {"exact_match": round(100 * em_total / n, 2), "f1": round(100 * f1_total / n, 2)}

In [ ]:
def qa_logits_over(model, features, batch_size=32):
    n = features["input_ids"].shape[0]
    starts, ends = [], []
    for i in range(0, n, batch_size):
        batch = {k: tf.constant(v[i:i + batch_size]) for k, v in features.items()}
        out = model(batch, training=False)
        starts.append(out.start_logits.numpy())
        ends.append(out.end_logits.numpy())
    return np.concatenate(starts), np.concatenate(ends)


lk.banner("scoring teacher and student")
ts, te = qa_logits_over(teacher, eval_feats)
ss, se = qa_logits_over(student, eval_feats)

teacher_preds = postprocess(eval_examples, eval_example_ids, eval_offsets, ts, te)
student_preds = postprocess(eval_examples, eval_example_ids, eval_offsets, ss, se)

teacher_score = score(teacher_preds, eval_examples)
student_score = score(student_preds, eval_examples)
print("teacher:", teacher_score)
print("student:", student_score)
print(f"\nF1 retained: {student_score['f1'] / teacher_score['f1']:.1%}")

### 7.1 Read the failures, not just the number

A metric tells you how much quality you lost. The examples tell you *what kind*. Run the next cell and classify each disagreement:

- **boundary error** — right region, wrong span edges (usually recoverable with more distillation steps or a longer `max_answer_len`)
- **distractor error** — a plausible but wrong entity of the correct type (the student lost the teacher's discrimination between similar candidates)
- **collapse** — an answer from the wrong part of the passage entirely (the student is too small, or under-trained)

Which category dominates determines what you would do next: more steps, a bigger student, or intermediate-layer distillation.

In [ ]:
shown = 0
for ex in eval_examples:
    tp, sp = teacher_preds[ex["id"]], student_preds[ex["id"]]
    if normalize_answer(tp) != normalize_answer(sp):
        print("Q      :", ex["question"])
        print("  gold :", ex["answers"]["text"][0])
        print("  teach :", tp)
        print("  stud  :", sp)
        print("  token F1 student vs gold:",
              round(max(f1(sp, g) for g in ex["answers"]["text"]), 2))
        print()
        shown += 1
    if shown >= 6:
        break

---
## 8. Measure and record

Quality is one axis. The reason we did this at all is the other two.

In [ ]:
bs1 = {k: tf.constant(v[:1]) for k, v in eval_feats.items()}

@tf.function(reduce_retracing=True)
def teacher_fn(x):
    o = teacher(x, training=False); return o.start_logits, o.end_logits

@tf.function(reduce_retracing=True)
def student_fn(x):
    o = student(x, training=False); return o.start_logits, o.end_logits

t_lat = lk.measure_latency(teacher_fn, bs1, warmup=10, runs=50, batch_size=1)
s_lat = lk.measure_latency(student_fn, bs1, warmup=10, runs=50, batch_size=1)
print("teacher:", t_lat)
print("student:", s_lat)
print(f"\nspeedup at batch 1: {t_lat['p50_ms'] / s_lat['p50_ms']:.2f}x "
      f"(predicted ~{teacher.config.num_hidden_layers / student_config.num_hidden_layers:.1f}x "
      f"from depth alone)")

The measured speedup will be **less** than the depth ratio predicts. That gap is the lesson:

- The embedding lookup, the QA head, and the Python/graph dispatch overhead are all fixed costs that do not shrink with depth.
- At batch size 1 the GPU is latency-bound rather than throughput-bound, so a smaller model does not keep it any busier.

This is exactly why the case study's biggest win came from quantization on **CPU**, not from distillation on GPU. Architecture reduction pays off most when it is combined with a change of serving substrate — which is the Day 2 argument.

In [ ]:
student_dir = lk.model_dir("student-squad-4L", clean=True)
student.save_pretrained(student_dir)
tokenizer.save_pretrained(student_dir)

teacher_dir = lk.model_dir("teacher-squad-12L", clean=True)
teacher.save_pretrained(teacher_dir)

t_size, s_size = lk.size_mb(teacher_dir), lk.size_mb(student_dir)

lk.record("squad-teacher",
          model="bert-base-uncased (SQuAD v1.1)", task="extractive QA", dataset="SQuAD v1.1",
          params_m=round(teacher.num_parameters()/1e6, 1), size_mb=round(t_size, 1),
          quality=teacher_score["f1"], quality_metric="F1",
          p50_ms=t_lat["p50_ms"], p95_ms=t_lat["p95_ms"],
          throughput_rps=t_lat["throughput_rps"], device=lk.device_label(),
          notes="12 layers; distillation teacher")

lk.record("distilled-4L",
          model="BERT 4-layer student", task="extractive QA", dataset="SQuAD v1.1",
          params_m=round(student.num_parameters()/1e6, 1), size_mb=round(s_size, 1),
          quality=student_score["f1"], quality_metric="F1",
          p50_ms=s_lat["p50_ms"], p95_ms=s_lat["p95_ms"],
          throughput_rps=s_lat["throughput_rps"], device=lk.device_label(),
          notes=f"T={TEMPERATURE}, alpha={ALPHA}, layers copied {LAYER_MAP}")

lk.ledger_df()

---
## 9. Package the technique for reuse

Lab 5 applies distillation to the **sentiment** model from Lab 1. Rather than rewriting the loss there, we save the reusable pieces now. This is also how you would want it in a real codebase: the KD machinery is task-independent, and only the loss head changes.

In [ ]:
KDKIT_SRC = r'''
"""kdkit.py - reusable knowledge-distillation utilities."""

import numpy as np
import tensorflow as tf

NEG_INF = -1e4


def soft_cross_entropy(teacher_logits, student_logits, T, mask=None):
    """Cross-entropy between temperature-softened teacher and student
    distributions. Optionally masks padded positions."""
    if mask is not None:
        penalty = (1.0 - tf.cast(mask, tf.float32)) * (-NEG_INF)
        teacher_logits = teacher_logits - penalty
        student_logits = student_logits - penalty
    t_probs = tf.nn.softmax(teacher_logits / T, axis=-1)
    s_logp = tf.nn.log_softmax(student_logits / T, axis=-1)
    return -tf.reduce_sum(t_probs * s_logp, axis=-1)


def make_classification_kd_loss(T=3.0, alpha=0.7):
    """Return loss_fn(model, batch, training) for single-label classification.

    Expected batch layout:
        (features, {"label": int32[B], "t_logits": float32[B, C]})
    """
    hard = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction=tf.keras.losses.Reduction.NONE
    )

    def loss_fn(model, batch, training):
        features, targets = batch
        logits = model(features, training=training).logits
        soft = soft_cross_entropy(targets["t_logits"], logits, T)
        hard_term = hard(targets["label"], logits)
        return tf.reduce_mean(alpha * (T**2) * soft + (1.0 - alpha) * hard_term)

    return loss_fn


def evenly_spaced_layers(n_teacher, n_student):
    """Teacher layer indices to copy into a student: evenly spaced, and always
    including the teacher's final layer."""
    step = n_teacher / n_student
    return [min(n_teacher - 1, int(round((i + 1) * step)) - 1) for i in range(n_student)]


def copy_encoder_weights(student, teacher, layer_map, encoder_attr="bert"):
    """Initialise a shallower student from a teacher of identical width."""
    s_main = getattr(student, encoder_attr)
    t_main = getattr(teacher, encoder_attr)
    s_main.embeddings.set_weights(t_main.embeddings.get_weights())
    for s_idx, t_idx in enumerate(layer_map):
        s_main.encoder.layer[s_idx].set_weights(t_main.encoder.layer[t_idx].get_weights())
    return layer_map


def cache_logits(model, features, batch_size=32, attr="logits"):
    """One teacher pass over a fixed dataset, for offline distillation."""
    n = features["input_ids"].shape[0]
    chunks = []
    for i in range(0, n, batch_size):
        batch = {k: tf.constant(v[i : i + batch_size]) for k, v in features.items()}
        chunks.append(getattr(model(batch, training=False), attr).numpy())
    return np.concatenate(chunks)
'''

from pathlib import Path
Path(ROOT, 'kdkit.py').write_text(KDKIT_SRC)

import kdkit, importlib
importlib.reload(kdkit)
print('kdkit written to', Path(ROOT, 'kdkit.py'))
print('12 -> 4 layer map:', kdkit.evenly_spaced_layers(12, 4))
print('12 -> 6 layer map:', kdkit.evenly_spaced_layers(12, 6))

---
## 10. Exercises

Work through at least the first two. They change parameters you have already wired up, so each is a re-run rather than a rewrite.

**1. Temperature sweep (no retraining).**
Reuse the cached teacher logits. For $T \in \{1, 2, 3, 5, 10\}$, compute the mean entropy of the teacher's start distribution over a few hundred training features. Plot entropy against $T$. At which point does the distribution carry less usable structure than the hard label, and how does that square with our choice of $T=3$?

**2. Depth ablation.**
Re-run Sections 5 and 6 with `LAYER_MAP = [1, 3, 5, 7, 9, 11]` (a 6-layer student). Record it as ledger stage `distilled-6L`. Plot F1 against p50 latency for the 12-, 6- and 4-layer models. Is the curve linear? Where would you set the operating point if the SLO were 25 ms and the quality floor were 90% of teacher F1?

**3. Initialisation ablation.**
Set `LAYER_MAP = [0, 1, 2, 3]` — the first four layers instead of evenly spaced ones — and retrain. Quantify the difference. This is the cheapest experiment here and usually the most surprising.

**4. Alpha ablation.**
Train with `ALPHA = 0.0` (hard labels only — ordinary fine-tuning of a small model) and `ALPHA = 1.0` (teacher only). The gap between `ALPHA = 0.0` and your `ALPHA = 0.7` run is the measured value of distillation as a technique, on your data, in your setup. It is the number to bring to a review.

**5. Intermediate-layer distillation (extension).**
Add a term that aligns the student's final hidden state with the teacher's, using cosine distance. `TFBertForQuestionAnswering` accepts `output_hidden_states=True`. Does it help at 4 layers more than at 6?

---
## 11. Wrap-up

### What you produced

- `models/student-squad-4L/` — a 4-layer QA student with its measured F1, latency and size in the ledger.
- `kdkit.py` — reusable KD utilities, applied again in Lab 5.
- Two more ledger rows.

### What to carry forward

1. **Distillation is the only lever that changes the architecture.** Everything after this operates on whatever shape you leave here, which is why it goes first.
2. **The embedding table does not shrink with depth.** For a 4-layer student it is nearly half the parameters. Quantization (Lab 3) hits it; pruning (Lab 4) usually should not.
3. **Offline distillation is free performance** when the training set is fixed. Cache the teacher's logits, drop the teacher from memory, iterate on the student cheaply.
4. **The speedup was smaller than the depth ratio.** Fixed overheads dominate at batch size 1. Keep that in mind when someone promises a 3× win from a 3× smaller model.

### Checkpoint

- [ ] `lk.ledger_df()` shows `squad-teacher` and `distilled-4L`.
- [ ] Student F1 is within roughly 10 points of the teacher's.
- [ ] You can state, from your own numbers, the F1 cost per millisecond saved.

### Next

**Lab 3 — Model Quantization.** We stop changing the architecture and start changing the *numbers*: fp32 → fp16 → int8. Per the syllabus we work on **IMDB** sentiment, and we do something the case study makes worth doing — quantize the Lab 1 SST-2 model and evaluate it on IMDB's much longer reviews, to test whether the precision loss survives a shift in input distribution.